# Citrus disease classification: clean, reproducible pipeline (publication outputs)

This notebook replaces `citrus-dataset-explore-using-ml-dl-models.ipynb`. It runs **one clean pipeline** and writes every number, table and figure used in the paper to `OUT_DIR`, so the notebook does not have to be run again to regenerate them.

**What it does**
1. Builds the dataset manifest (fruit + leaf images, six classes) and a *stratified* 70/15/15 split (seed 42). Optional: stratified k-fold cross-validation.
2. Fine-tunes four ImageNet-pretrained CNNs (EfficientNetB0, ResNet50, DenseNet121, InceptionV3) with the same two-phase procedure as before, class weights and augmentation. Phase 1 (frozen) results are also stored, so the effect of fine-tuning can be reported.
3. Trains SVM / Random Forest / KNN / Naive Bayes on frozen ImageNet features (ResNet50, DenseNet121) and, optionally, on fine-tuned features (StandardScaler -> PCA(256) -> classifier, fitted on the training split only).
4. Builds the ResNet50 + DenseNet121 soft-voting ensemble from the saved test probabilities.
5. Saves predictions, metrics, confusion matrices, histories, feature files, LaTeX tables and 600-dpi figures (PNG + vector PDF), plus organ-wise accuracy and Grad-CAM.

**Fixes compared with the old notebook**
- one EfficientNetB0 run only (the old notebook trained it twice) and no duplicated plotting/metric cells;
- the best weights are reloaded explicitly after each phase (Keras `EarlyStopping(restore_best_weights=True)` only restores weights when early stopping triggers; the old notebook evaluated the in-memory model);
- Phase 1 and Phase 2 use separate checkpoints, and the final model is the phase with the lower validation loss (the old notebook let Phase 2 overwrite the Phase 1 checkpoint);
- all seeds are set in one place (Python, NumPy, TensorFlow), the file list is sorted, and every metric is computed in one function;
- every stage is cached: if its output files exist the stage is skipped (`FORCE_RETRAIN = False`).

**How to use**: set the flags in the configuration cell, then *Run All*. Notes:
- Kaggle: switch **Internet ON** (ImageNet weights are downloaded by Keras) and attach the citrus dataset.
- Use `SMOKE_TEST = True` first for a dry run on synthetic images (about 8 minutes on CPU, no dataset needed). Its numbers are meaningless; it only checks that every cell and output file works.
- Rough run time on a Kaggle T4 (estimate, not measured): about 30-60 minutes for the hold-out run; cross-validation multiplies this by the number of folds.
- GPU training is not bit-wise deterministic, so repeated runs can differ by one or two test images even with fixed seeds.

In [ ]:
import os, re, gc, json, time, math, random, zipfile, shutil, platform, warnings, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, matthews_corrcoef, precision_recall_fscore_support)
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPUs:", [g.name for g in tf.config.list_physical_devices("GPU")])

## 1. Configuration (the only cell you normally edit)

In [ ]:
# ----------------------------------------------------------------------------------------
# User switches
# ----------------------------------------------------------------------------------------
SMOKE_TEST       = False   # True = tiny synthetic dataset, random weights, 1+1 epochs (pipeline check only)
FORCE_RETRAIN    = False   # True = ignore cached files and recompute everything
RUN_CV           = False   # True = also run stratified k-fold cross-validation (about 5x the training time)
CV_FOLDS         = 5
CV_RUN_FOLDS     = None    # e.g. [4, 5] = train only these folds (1-based); None = all folds. Finished folds are never recomputed.
RESTORE_FROM     = None    # e.g. "/kaggle/input/<earlier-notebook-output>/results": copy earlier results into OUT_DIR to resume a run
RUN_FINETUNED_FEATURES = True   # ML classifiers on fine-tuned CNN features (cheap ablation)
RUN_GRADCAM      = True    # Grad-CAM figure for the final ResNet50 (inference only)

CFG = dict(
    SEED = 42,
    DATA_ROOT   = "/kaggle/input",                # searched recursively for a folder called Citrus_dataset
    DATASET_DIR = None,                           # or set explicitly, e.g. "/kaggle/input/<name>/Citrus_dataset"
    OUT_DIR     = "/kaggle/working/results",
    CLASSES     = ["Black_spot", "Canker", "Greening", "Healthy", "Melanose", "Scab"],
    MODELS      = ["EfficientNetB0", "ResNet50", "DenseNet121", "InceptionV3"],
    IMG         = {"EfficientNetB0": (224, 224), "ResNet50": (224, 224), "DenseNet121": (224, 224), "InceptionV3": (299, 299)},
    BATCH       = {"EfficientNetB0": 32, "ResNet50": 32, "DenseNet121": 32, "InceptionV3": 24},
    DROPOUT     = {"EfficientNetB0": 0.2, "ResNet50": 0.4, "DenseNet121": 0.4, "InceptionV3": 0.4},
    WEIGHTS     = "imagenet",                     # None only for the smoke test
    LR1 = 1e-3, LR2 = 1e-5, EPOCHS1 = 20, EPOCHS2 = 20, N_UNFREEZE = 60,
    ES_PATIENCE = 8, RLR_FACTOR = 0.2, RLR_PATIENCE = 8, MIN_LR = 1e-6,
    PCA_COMPONENTS = 256,
    RF_TREES = {"ResNet50": 600, "DenseNet121": 500},   # as described in the paper
    ML_BACKBONES = ["ResNet50", "DenseNet121"],          # frozen ImageNet backbones used as feature extractors
    ML_IMG = (224, 224), ML_BATCH = 32,
    FIG_DPI = 600,
    GRADCAM_MODEL = "ResNet50",
)

if SMOKE_TEST:
    CFG.update(OUT_DIR="/tmp/citrus_smoke/results", DATASET_DIR="/tmp/citrus_smoke/Citrus_dataset", WEIGHTS=None,
               EPOCHS1=1, EPOCHS2=1, N_UNFREEZE=10, FIG_DPI=150, ML_IMG=(96, 96), ML_BATCH=16,
               IMG={m: ((96, 96) if m != "InceptionV3" else (96, 96)) for m in CFG["MODELS"]},
               BATCH={m: 16 for m in CFG["MODELS"]}, PCA_COMPONENTS=16)
    RUN_CV, CV_FOLDS, FORCE_RETRAIN = True, 2, True

CLASSES = CFG["CLASSES"]; C2I = {c: i for i, c in enumerate(CLASSES)}; SEED = CFG["SEED"]
OUT = Path(CFG["OUT_DIR"]); OUT.mkdir(parents=True, exist_ok=True)
for sub in ("data", "paper_assets/figures", "paper_assets/tables"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)
if RESTORE_FROM and Path(RESTORE_FROM).exists():            # resume after an interrupted or failed run
    shutil.copytree(RESTORE_FROM, OUT, dirs_exist_ok=True); print("Restored earlier results from", RESTORE_FROM)

def set_seed(seed=SEED):
    # Python, NumPy and TensorFlow in one call
    keras.utils.set_random_seed(seed)
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

set_seed()
print("Output folder:", OUT)

## 2. Dataset manifest (fruit + leaf images, six classes)

In [ ]:
def normalize_label(folder_name):
    s = folder_name.strip().lower()
    s = re.sub(r"[\s\-]+", "_", s)
    s = re.sub(r"[^a-z_]+", "", s)
    if s == "blackspot":
        s = "black_spot"
    return {"black_spot": "Black_spot", "canker": "Canker", "greening": "Greening",
            "healthy": "Healthy", "melanose": "Melanose", "scab": "Scab"}.get(s)

def find_dataset_dir():
    if CFG["DATASET_DIR"]:
        return CFG["DATASET_DIR"]
    for root, dirs, _ in os.walk(CFG["DATA_ROOT"]):
        if os.path.basename(root) == "Citrus_dataset":
            return root
    raise FileNotFoundError("Citrus_dataset not found under DATA_ROOT; set CFG['DATASET_DIR'] manually.")

def collect_images(dataset_dir):
    rows = []
    for part in ("Fruits", "Leaves"):                       # 'domain' = organ
        part_dir = os.path.join(dataset_dir, part)
        if not os.path.isdir(part_dir):
            continue
        for cls_folder in sorted(os.listdir(part_dir)):     # sorted -> deterministic order
            cls_dir = os.path.join(part_dir, cls_folder)
            label = normalize_label(cls_folder) if os.path.isdir(cls_dir) else None
            if label is None:
                continue
            for fn in sorted(os.listdir(cls_dir)):
                if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                    rows.append({"filepath": os.path.join(cls_dir, fn), "label": label, "domain": part})
    return pd.DataFrame(rows)

In [ ]:
def make_synthetic_dataset(root, seed=0):
    # Tiny fake dataset with the same folder layout (smoke test only; the images carry no real information).
    from PIL import Image
    rng = np.random.default_rng(seed)
    plan = {"Black_spot": {"Fruits": 6, "Leaves": 10}, "Canker": {"Fruits": 8, "Leaves": 10},
            "Greening": {"Fruits": 4, "Leaves": 14}, "Healthy": {"Fruits": 5, "Leaves": 9},
            "Melanose": {"Leaves": 12}, "Scab": {"Fruits": 12}}
    tint = {c: rng.integers(40, 215, 3) for c in plan}
    for cls, parts in plan.items():
        for part, n in parts.items():
            d = Path(root) / part / cls; d.mkdir(parents=True, exist_ok=True)
            for k in range(n):
                arr = np.clip(rng.normal(tint[cls], 35, (96, 96, 3)), 0, 255).astype(np.uint8)
                Image.fromarray(arr).save(d / f"{cls}_{k:03d}.jpg")

if SMOKE_TEST:
    if not Path(CFG["DATASET_DIR"]).exists():
        make_synthetic_dataset(CFG["DATASET_DIR"])
    print("Synthetic dataset created at", CFG["DATASET_DIR"])

DATASET_DIR = find_dataset_dir()
df = collect_images(DATASET_DIR)
assert len(df) > 0, "No images found. Check DATASET_DIR."
assert set(df["label"]) == set(CLASSES), f"Missing classes: {set(CLASSES) - set(df['label'])}"
df.to_csv(OUT / "data" / "dataset_manifest.csv", index=False)

counts = df.groupby(["label", "domain"]).size().unstack(fill_value=0).reindex(CLASSES)
counts["Total"] = counts.sum(axis=1); counts["Percent"] = (100 * counts["Total"] / counts["Total"].sum()).round(1)
counts.to_csv(OUT / "data" / "class_organ_counts.csv")
print("Dataset:", DATASET_DIR, "| images:", len(df))
display(counts)

## 3. Splits (stratified) and class weights
The hold-out split is the one used in the paper (70/15/15, `random_state=42`, stratified by class). Because file names are now sorted, the exact images in each subset may differ from the old notebook, so **all numbers must be taken from this run**. The split is saved to `data/split_manifest.csv`.

In [ ]:
def holdout_split(frame, seed=SEED):
    train_df, temp_df = train_test_split(frame, test_size=0.30, random_state=seed, stratify=frame["label"])
    val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=seed, stratify=temp_df["label"])
    return [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]

def cv_splits(frame, k, seed=SEED):
    # outer: stratified k-fold (test); inner: 15 % of the remaining images become the validation set
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    for fold, (tr_idx, te_idx) in enumerate(skf.split(frame, frame["label"])):
        rest, test = frame.iloc[tr_idx], frame.iloc[te_idx]
        train, val = train_test_split(rest, test_size=0.15, random_state=seed, stratify=rest["label"])
        yield fold, train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

def class_weight_dict(train_frame):
    y = train_frame["label"].map(C2I).values
    w = compute_class_weight("balanced", classes=np.arange(len(CLASSES)), y=y)   # N / (K * n_c)
    return {int(i): float(v) for i, v in enumerate(w)}

train_df, val_df, test_df = holdout_split(df)
assert len(train_df) + len(val_df) + len(test_df) == len(df)
split_manifest = pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")])
split_manifest.to_csv(OUT / "data" / "split_manifest.csv", index=False)
split_counts = (split_manifest.groupby(["label", "split"]).size().unstack(fill_value=0)
                .reindex(index=CLASSES, columns=["train", "val", "test"]))
split_counts.loc["Total"] = split_counts.sum()
split_counts.to_csv(OUT / "data" / "split_counts.csv")
display(split_counts)

cw = class_weight_dict(train_df)
json.dump({CLASSES[i]: w for i, w in cw.items()}, open(OUT / "data" / "class_weights.json", "w"), indent=2)
print("class weights:", {CLASSES[i]: round(w, 2) for i, w in cw.items()})

## 4. Helpers: generators, models, metrics, saving

In [ ]:
APPS = {   # constructor + the matching preprocess_input
    "EfficientNetB0": (keras.applications.EfficientNetB0, keras.applications.efficientnet.preprocess_input),
    "ResNet50":       (keras.applications.ResNet50,       keras.applications.resnet.preprocess_input),
    "DenseNet121":    (keras.applications.DenseNet121,    keras.applications.densenet.preprocess_input),
    "InceptionV3":    (keras.applications.InceptionV3,    keras.applications.inception_v3.preprocess_input),
}

def log_mem(tag=""):
    # host-memory log line (helps to diagnose a kernel that dies during a long run)
    try:
        import psutil
        print(f"[mem] {tag}: process RSS = {psutil.Process().memory_info().rss / 1e9:.1f} GB, free RAM = {psutil.virtual_memory().available / 1e9:.1f} GB")
    except Exception:
        pass

def slug(text):
    return re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")

def run_dir(tag):
    root = OUT / tag
    for s in ("models", "history", "features", "preds", "metrics", "tables", "figures"):
        (root / s).mkdir(parents=True, exist_ok=True)
    return root

_FLOW = dict(x_col="filepath", y_col="label", class_mode="categorical")

def make_gens(tr_df, va_df, te_df, img_size, batch, pre):
    # Augmentation only on the training images (same parameters as in the paper, Table 'tab:aug')
    aug = ImageDataGenerator(preprocessing_function=pre, rotation_range=25, width_shift_range=0.15,
                             height_shift_range=0.15, shear_range=0.10, zoom_range=0.20, horizontal_flip=True,
                             brightness_range=(0.85, 1.15), channel_shift_range=10.0, fill_mode="nearest")
    plain = ImageDataGenerator(preprocessing_function=pre)
    kw = dict(target_size=img_size, classes=CLASSES, **_FLOW)
    tr = aug.flow_from_dataframe(tr_df, batch_size=batch, shuffle=True, seed=SEED, **kw)
    va = plain.flow_from_dataframe(va_df, batch_size=batch, shuffle=False, **kw)
    te = plain.flow_from_dataframe(te_df, batch_size=batch, shuffle=False, **kw)
    for g, f in ((tr, tr_df), (va, va_df), (te, te_df)):
        assert g.samples == len(f), "some image files could not be read"
    return tr, va, te

def make_plain_gen(frame, img_size, batch, pre):
    g = ImageDataGenerator(preprocessing_function=pre).flow_from_dataframe(
        frame, target_size=img_size, classes=CLASSES, batch_size=batch, shuffle=False, **_FLOW)
    assert g.samples == len(frame)
    return g

def build_model(name, img_size, dropout, weights):
    ctor, _ = APPS[name]
    base = ctor(include_top=False, weights=weights, input_shape=(img_size[0], img_size[1], 3))
    inputs = keras.Input(shape=(img_size[0], img_size[1], 3))
    x = base(inputs, training=False)               # base always runs in inference mode (BatchNorm statistics frozen)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout, name="dropout")(x)
    outputs = layers.Dense(len(CLASSES), activation="softmax", name="pred")(x)
    return keras.Model(inputs, outputs, name=f"{name}_clf"), base

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss="categorical_crossentropy", metrics=["accuracy"])

def unfreeze_last_n(base, n):
    base.trainable = True
    for layer in base.layers[:-n]:
        layer.trainable = False

def make_callbacks(ckpt_path):
    return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=CFG["ES_PATIENCE"], restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=CFG["RLR_FACTOR"], patience=CFG["RLR_PATIENCE"], min_lr=CFG["MIN_LR"]),
            keras.callbacks.ModelCheckpoint(str(ckpt_path), monitor="val_loss", save_best_only=True, save_weights_only=True)]

def compute_all_metrics(y_true, y_pred):
    labels = list(range(len(CLASSES)))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    return {"Accuracy": float(accuracy_score(y_true, y_pred)),
            "Macro_Precision": float(p.mean()), "Macro_Recall": float(r.mean()), "Macro_F1": float(f.mean()),
            "Weighted_F1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
            "MCC": float(matthews_corrcoef(y_true, y_pred)),
            "per_class": {c: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]), "support": int(s[i])} for i, c in enumerate(CLASSES)},
            "confusion_matrix": cm.tolist(), "n_test": int(len(y_true))}

def save_result(tag, name, y_true, probs, test_frame, group, features):
    root = run_dir(tag); y_pred = np.asarray(probs).argmax(1)
    m = compute_all_metrics(y_true, y_pred); m.update(name=name, group=group, features=features)
    json.dump(m, open(root / "metrics" / f"{slug(name)}.json", "w"), indent=2)
    out = test_frame[["filepath", "domain", "label"]].copy().reset_index(drop=True)
    out["y_true"], out["y_pred"] = y_true, y_pred
    for j, c in enumerate(CLASSES):
        out[f"p_{c}"] = np.asarray(probs)[:, j]
    out.to_csv(root / "preds" / f"{slug(name)}.csv", index=False)
    print(f"  saved {name:<46s} acc={m['Accuracy']:.4f}  macroF1={m['Macro_F1']:.4f}")
    return m

def load_preds(tag, name):
    d = pd.read_csv(OUT / tag / "preds" / f"{slug(name)}.csv")
    return d["y_true"].values, d[[f"p_{c}" for c in CLASSES]].values, d

def done(tag, name):
    return (not FORCE_RETRAIN) and (OUT / tag / "metrics" / f"{slug(name)}.json").exists()

# warn if cached results were produced with another configuration
cfg_file = OUT / "config.json"
cfg_now = json.loads(json.dumps(CFG, default=str))
if cfg_file.exists() and json.load(open(cfg_file)) != cfg_now and not FORCE_RETRAIN:
    print("WARNING: CFG differs from the cached run. Set FORCE_RETRAIN = True (or delete OUT_DIR) to recompute.")
json.dump(cfg_now, open(cfg_file, "w"), indent=2)

## 5. Deep learning models: two-phase fine-tuning (one function, used for all four networks)
Phase 1 trains the new head on the frozen base; Phase 2 unfreezes the last 60 layers (learning rate 1e-5). After each phase the **best-validation-loss checkpoint is reloaded**; the final model is the phase with the lower validation loss. The frozen-only (Phase 1) test result is stored too.

In [ ]:
def train_or_load_dl(tag, name, tr_df, va_df, te_df, class_weights):
    root = run_dir(tag)
    final_path, hist_path = root / "models" / f"{name}_final.keras", root / "history" / f"{name}_history.csv"
    if done(tag, name) and done(tag, f"{name} (frozen, Phase 1 only)") and hist_path.exists():
        print(f"[cache] {name}"); return
    print(f"\n=== {name} ===")
    set_seed(SEED); keras.backend.clear_session(); gc.collect()
    img, bs, drop = CFG["IMG"][name], CFG["BATCH"][name], CFG["DROPOUT"][name]
    _, pre = APPS[name]
    tr, va, te = make_gens(tr_df, va_df, te_df, img, bs, pre)
    model, base = build_model(name, img, drop, CFG["WEIGHTS"])
    ck1, ck2 = root / "models" / f"{name}_phase1.weights.h5", root / "models" / f"{name}_phase2.weights.h5"
    t0 = time.time()

    base.trainable = False; compile_model(model, CFG["LR1"])
    h1 = model.fit(tr, validation_data=va, epochs=CFG["EPOCHS1"], class_weight=class_weights, callbacks=make_callbacks(ck1), verbose=2)
    model.load_weights(ck1)                                                       # best Phase-1 weights
    save_result(tag, f"{name} (frozen, Phase 1 only)", te.classes, model.predict(te, verbose=0), te_df, "DL-phase1", "frozen")

    unfreeze_last_n(base, CFG["N_UNFREEZE"]); compile_model(model, CFG["LR2"])
    h2 = model.fit(tr, validation_data=va, epochs=CFG["EPOCHS2"], class_weight=class_weights, callbacks=make_callbacks(ck2), verbose=2)
    best1, best2 = min(h1.history["val_loss"]), min(h2.history["val_loss"])
    chosen = "phase2" if best2 < best1 else "phase1"
    model.load_weights(ck2 if chosen == "phase2" else ck1)                         # best weights of the chosen phase
    model.save(final_path)
    save_result(tag, name, te.classes, model.predict(te, verbose=0), te_df, "DL", "fine-tuned")

    rows = []
    for ph, h in ((1, h1), (2, h2)):
        d = pd.DataFrame(h.history); d.insert(0, "epoch_in_phase", np.arange(1, len(d) + 1)); d.insert(0, "phase", ph); rows.append(d)
    pd.concat(rows, ignore_index=True).to_csv(hist_path, index=False)
    json.dump({"model": name, "epochs_phase1": len(h1.history["val_loss"]), "epochs_phase2": len(h2.history["val_loss"]),
               "best_val_loss_phase1": float(best1), "best_val_loss_phase2": float(best2), "final_model_from": chosen,
               "train_seconds": round(time.time() - t0, 1), "image_size": list(img), "batch_size": bs, "dropout": drop},
              open(root / "history" / f"{name}_train_info.json", "w"), indent=2)
    del model, base, tr, va, te; keras.backend.clear_session(); gc.collect(); log_mem(f"{tag}/{name}")

## 6. Machine-learning classifiers on deep features (frozen ImageNet backbone; optionally fine-tuned features)
Pipeline: `StandardScaler -> PCA(256) -> classifier`, fitted on the **training features only** and evaluated once on the test features.

In [ ]:
def extract_features(feature_model, gen):
    f = feature_model.predict(gen, verbose=0)
    f = np.concatenate(f, axis=0) if isinstance(f, list) else np.asarray(f)
    assert len(f) == len(gen.classes)
    return f.astype(np.float32), np.asarray(gen.classes, dtype=np.int64)

def features_for(tag, key, make_feature_model, pre, img, frames):
    path = run_dir(tag) / "features" / f"{key}.npz"
    if path.exists() and not FORCE_RETRAIN:
        z = np.load(path); return {s: (z[f"X_{s}"], z[f"y_{s}"]) for s in ("train", "val", "test")}
    feature_model = make_feature_model()          # built only when the features are not cached
    out = {}
    for s, fr in zip(("train", "val", "test"), frames):
        out[s] = extract_features(feature_model, make_plain_gen(fr, img, CFG["ML_BATCH"], pre))
    np.savez_compressed(path, **{f"{k}_{s}": out[s][i] for s in out for i, k in enumerate(("X", "y"))})
    del feature_model; keras.backend.clear_session(); gc.collect(); log_mem(f"{tag}/{key} features")   # release the extractor
    return out

def frozen_features(tag, backbone, frames):
    ctor, pre = APPS[backbone]; img = CFG["ML_IMG"]
    def make():
        base = ctor(include_top=False, weights=CFG["WEIGHTS"], input_shape=(img[0], img[1], 3)); base.trainable = False
        inp = keras.Input(shape=(img[0], img[1], 3)); x = layers.GlobalAveragePooling2D()(base(inp, training=False))
        return keras.Model(inp, x)
    return features_for(tag, f"{backbone}_frozen", make, pre, img, frames)

def finetuned_features(tag, backbone, frames):
    def make():
        model = keras.models.load_model(run_dir(tag) / "models" / f"{backbone}_final.keras")
        gap = [l for l in model.layers if isinstance(l, layers.GlobalAveragePooling2D)][0]
        return keras.Model(model.input, gap.output)
    _, pre = APPS[backbone]
    return features_for(tag, f"{backbone}_finetuned", make, pre, CFG["IMG"][backbone], frames)

def build_ml_pipelines(X_train, y_train, backbone):
    n_comp = max(1, min(CFG["PCA_COMPONENTS"], X_train.shape[0] - 1, X_train.shape[1]))
    counts = np.bincount(y_train, minlength=len(CLASSES)).astype(float)
    priors = (counts + 1e-6) / (counts + 1e-6).sum()
    pk = dict(n_components=n_comp, random_state=SEED)
    if X_train.shape[1] > 2000:
        pk["svd_solver"] = "randomized"
    def pipe(clf):
        return Pipeline([("scaler", StandardScaler()), ("pca", PCA(**pk)), ("clf", clf)])
    return {"SVM": pipe(SVC(kernel="rbf", C=10.0, gamma="scale", class_weight="balanced", probability=True, random_state=SEED)),
            "RF":  pipe(RandomForestClassifier(n_estimators=CFG["RF_TREES"][backbone], random_state=SEED, n_jobs=-1, class_weight="balanced_subsample")),
            "KNN": pipe(KNeighborsClassifier(n_neighbors=5, weights="distance", n_jobs=-1)),
            "NB":  pipe(GaussianNB(var_smoothing=1e-3, priors=priors))}

def run_ml(tag, backbone, source, feats, te_df):
    (Xtr, ytr), _, (Xte, yte) = feats["train"], feats["val"], feats["test"]
    for clf_name, pipe in build_ml_pipelines(Xtr, ytr, backbone).items():
        name = f"{backbone} ({source}) + {clf_name}"
        if done(tag, name):
            continue
        pipe.fit(Xtr, ytr)
        group = "Hybrid DL+ML" if (source == "frozen" and backbone == "DenseNet121" and clf_name == "RF") else "ML"
        save_result(tag, name, yte, pipe.predict_proba(Xte), te_df, group, source)

def run_ensemble(tag, te_df):
    name = "ResNet50 + DenseNet121 (soft voting)"
    if done(tag, name):
        return
    y1, p1, _ = load_preds(tag, "ResNet50"); y2, p2, _ = load_preds(tag, "DenseNet121")
    assert (y1 == y2).all(), "test order differs between the two models"
    save_result(tag, name, y1, (p1 + p2) / 2.0, te_df, "Hybrid DL+DL", "fine-tuned")

## 7. One experiment = one function (hold-out now, cross-validation folds later)

In [ ]:
def collect_summary(tag):
    rows = []
    for f in sorted((OUT / tag / "metrics").glob("*.json")):
        if f.name == "summary.json":
            continue
        m = json.load(open(f)); r = {k: m[k] for k in ("name", "group", "features", "Accuracy", "Macro_Precision", "Macro_Recall", "Macro_F1", "Weighted_F1", "MCC", "n_test")}
        for c in CLASSES:
            r[f"F1_{c}"] = m["per_class"][c]["f1"]; r[f"R_{c}"] = m["per_class"][c]["recall"]; r[f"P_{c}"] = m["per_class"][c]["precision"]
        rows.append(r)
    out = pd.DataFrame(rows).sort_values("name").reset_index(drop=True)
    out.to_csv(OUT / tag / "metrics" / "summary.csv", index=False)
    return out

def run_experiment(tag, tr_df, va_df, te_df, models=None):
    models = models or CFG["MODELS"]; run_dir(tag)
    cwd = class_weight_dict(tr_df)
    for name in models:
        train_or_load_dl(tag, name, tr_df, va_df, te_df, cwd)
    frames = (tr_df, va_df, te_df)
    for bb in CFG["ML_BACKBONES"]:
        print(f"\n--- ML on frozen {bb} features ---")
        run_ml(tag, bb, "frozen", frozen_features(tag, bb, frames), te_df)
        if RUN_FINETUNED_FEATURES and bb in models:
            print(f"--- ML on fine-tuned {bb} features ---")
            run_ml(tag, bb, "fine-tuned", finetuned_features(tag, bb, frames), te_df)
    if {"ResNet50", "DenseNet121"} <= set(models):
        print("\n--- soft-voting ensemble ---"); run_ensemble(tag, te_df)
    keras.backend.clear_session(); gc.collect()
    return collect_summary(tag)

## 8. Run the hold-out experiment (this is the experiment reported in the paper)
Training can take a while; every finished model is saved, so after an interruption simply re-run this cell.

In [ ]:
summary = run_experiment("holdout", train_df, val_df, test_df)
display(summary[["name", "group", "features", "Accuracy", "Macro_Precision", "Macro_Recall", "Macro_F1", "Weighted_F1"]].round(4))

## 9. Publication-quality figures and LaTeX tables (hold-out run)
Everything below only **reads the saved results**, so this section can be re-run in seconds without any training. Figures are written as PNG (`FIG_DPI` = 600) and vector PDF; the file names are the ones used in the paper (`paper_assets/figures`). Tables are `tabular` environments that can be pasted with `\input{tables/...}`.

In [ ]:
CLASS_LABELS = [c.replace("_", " ") for c in CLASSES]
OKABE = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#F0E442", "#000000"]

def set_pub_style():
    matplotlib.rcParams.update({
        "font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "Nimbus Roman", "DejaVu Serif"],
        "mathtext.fontset": "stix", "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
        "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8, "axes.linewidth": 0.8,
        "figure.dpi": 110, "savefig.dpi": CFG["FIG_DPI"], "savefig.bbox": "tight", "savefig.pad_inches": 0.05,
        "pdf.fonttype": 42, "ps.fonttype": 42, "axes.spines.top": False, "axes.spines.right": False,
    })
set_pub_style()

def save_fig(fig, name, tag="holdout"):
    # PNG at FIG_DPI + vector PDF, both in <tag>/figures and (for the hold-out run) in paper_assets/figures
    targets = [OUT / tag / "figures"] + ([OUT / "paper_assets" / "figures"] if tag in ("holdout", "cv_summary") else [])
    for d in targets:
        d.mkdir(parents=True, exist_ok=True)
        fig.savefig(d / f"{name}.png", dpi=CFG["FIG_DPI"]); fig.savefig(d / f"{name}.pdf")
    plt.show(); plt.close(fig)
    print("saved figure:", name)

def _ml_rows(backbone):
    return [(f"{backbone} + {c}", f"{backbone} (frozen) + {c}") for c in ("SVM", "KNN", "RF", "NB")]

PAPER_LAYOUT = [
    ("Deep learning", [("EfficientNetB0", "EfficientNetB0"), ("ResNet50", "ResNet50"), ("DenseNet121", "DenseNet121"), ("InceptionV3", "InceptionV3")]),
    ("DL+ML (ResNet50)", _ml_rows("ResNet50")),
    ("DL+ML (DenseNet121)", _ml_rows("DenseNet121")),
    ("Hybrid DL+DL", [("ResNet50 + DenseNet121", "ResNet50 + DenseNet121 (soft voting)")]),
]
PAPER_ROWS = [(lab, n) for _, items in PAPER_LAYOUT for lab, n in items]
METRICS = [("Accuracy", "Accuracy"), ("Macro_F1", "Macro F1"), ("Weighted_F1", "Weighted F1"), ("Macro_Recall", "Macro recall")]

def load_metrics(tag, name):
    return json.load(open(OUT / tag / "metrics" / f"{slug(name)}.json"))

def available_rows(tag):
    return [(lab, n) for lab, n in PAPER_ROWS if (OUT / tag / "metrics" / f"{slug(n)}.json").exists()]

In [ ]:
# ---- Figure: class distribution by organ -------------------------------------------------
def fig_class_distribution():
    cnt = counts.reindex(CLASSES)
    fig, ax = plt.subplots(figsize=(5.2, 2.9))
    x = np.arange(len(CLASSES)); fr = cnt.get("Fruits", pd.Series(0, index=CLASSES)); lf = cnt.get("Leaves", pd.Series(0, index=CLASSES))
    ax.bar(x, fr, color=OKABE[1], label="Fruit images", edgecolor="black", linewidth=0.5)
    ax.bar(x, lf, bottom=fr, color=OKABE[2], label="Leaf images", edgecolor="black", linewidth=0.5)
    for i, c in enumerate(CLASSES):
        ax.text(i, cnt.loc[c, "Total"] + 0.01 * cnt["Total"].max(), int(cnt.loc[c, "Total"]), ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(CLASS_LABELS, rotation=25, ha="right"); ax.set_ylabel("Number of images")
    ax.set_ylim(0, cnt["Total"].max() * 1.15); ax.legend(frameon=False); ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)
    save_fig(fig, "fig_class_dist")

# ---- Figure: training curves -------------------------------------------------------------
def fig_training_curves(tag="holdout"):
    models = [m for m in CFG["MODELS"] if (OUT / tag / "history" / f"{m}_history.csv").exists()]
    fig, axes = plt.subplots(2, len(models), figsize=(1.9 * len(models) + 0.6, 4.0), sharex="col")
    axes = np.array(axes).reshape(2, len(models))
    for j, m in enumerate(models):
        h = pd.read_csv(OUT / tag / "history" / f"{m}_history.csv"); h["epoch"] = np.arange(1, len(h) + 1)
        n1 = int((h["phase"] == 1).sum())
        for i, (tr, va, yl) in enumerate((("accuracy", "val_accuracy", "Accuracy"), ("loss", "val_loss", "Loss"))):
            ax = axes[i, j]
            ax.plot(h["epoch"], h[tr], color=OKABE[0], lw=1.2, label="Train")
            ax.plot(h["epoch"], h[va], color=OKABE[1], lw=1.2, label="Validation")
            ax.axvline(n1 + 0.5, color="grey", ls="--", lw=0.8)
            if j == 0: ax.set_ylabel(yl)
            if i == 0: ax.set_title(m, fontsize=9)
            if i == 1: ax.set_xlabel("Epoch")
            ax.grid(alpha=0.3)
    axes[0, 0].legend(frameon=False, loc="lower right")
    fig.text(0.5, -0.01, "Dashed line: start of Phase 2 (fine-tuning)", ha="center", fontsize=8)
    fig.tight_layout(); save_fig(fig, "fig_curves", tag)

# ---- Figure: confusion matrices ---------------------------------------------------------
def draw_cm(ax, cm, title):
    cm = np.array(cm); norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES)))
    ax.set_xticklabels(CLASS_LABELS, rotation=40, ha="right", fontsize=7); ax.set_yticklabels(CLASS_LABELS, fontsize=7)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i, j]}\n({100 * norm[i, j]:.0f}%)", ha="center", va="center", fontsize=6.3,
                    color="white" if norm[i, j] > 0.55 else "black")
    ax.set_title(title, fontsize=9, fontweight="bold"); ax.set_xlabel("Predicted label", fontsize=8); ax.set_ylabel("True label", fontsize=8)
    for s in ax.spines.values(): s.set_visible(True)
    return im

def cm_grid(items, ncols, fname, tag="holdout", width=7.0, aspect=0.92):
    items = [(t, n) for t, n in items if (OUT / tag / "metrics" / f"{slug(n)}.json").exists()]
    if not items: return
    nrows = math.ceil(len(items) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(width, width / ncols * aspect * nrows), squeeze=False)
    for ax in axes.ravel(): ax.axis("off")
    for ax, (title, name) in zip(axes.ravel(), items):
        ax.axis("on"); im = draw_cm(ax, load_metrics(tag, name)["confusion_matrix"], title)
    fig.tight_layout(); cb = fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.025, pad=0.02); cb.set_label("Row-normalised (recall)", fontsize=8)
    save_fig(fig, fname, tag)

# ---- Figure: comparison bars and per-class heat map -------------------------------------
def fig_comparison(tag="holdout"):
    rows = available_rows(tag)
    fig, ax = plt.subplots(figsize=(7.2, 3.3)); w = 0.2; x = np.arange(len(rows))
    for k, (key, lab) in enumerate(METRICS):
        ax.bar(x + (k - 1.5) * w, [100 * load_metrics(tag, n)[key] for _, n in rows], w, label=lab, color=OKABE[[0, 1, 2, 3][k]], edgecolor="black", linewidth=0.3)
    ax.set_xticks(x); ax.set_xticklabels([r[0] for r in rows], rotation=30, ha="right"); ax.set_ylabel("Score (%)"); ax.set_ylim(0, 100)
    ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False); ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)
    fig.tight_layout(); save_fig(fig, "fig_comparison", tag)

def fig_perclass_f1(tag="holdout"):
    rows = available_rows(tag)
    M = np.array([[load_metrics(tag, n)["per_class"][c]["f1"] for c in CLASSES] for _, n in rows])
    sup = [load_metrics(tag, rows[0][1])["per_class"][c]["support"] for c in CLASSES]
    fig, ax = plt.subplots(figsize=(6.2, 0.38 * len(rows) + 1.0)); im = ax.imshow(M, cmap="viridis", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels([f"{l}\n(n={s})" for l, s in zip(CLASS_LABELS, sup)], fontsize=8)
    ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows], fontsize=8)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if M[i, j] < 0.6 else "black")
    cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02); cb.set_label("F1-score", fontsize=8)
    for s in ax.spines.values(): s.set_visible(True)
    fig.tight_layout(); save_fig(fig, "fig_perclass_f1", tag)

In [ ]:
# ---- LaTeX tables (tabular only; paste with \input{tables/<file>.tex}) -------------------
def pct(v): return f"{100 * v:.1f}"

def write_tex(tag, fname, text):
    for d in [OUT / tag / "tables"] + ([OUT / "paper_assets" / "tables"] if tag in ("holdout", "cv_summary") else []):
        d.mkdir(parents=True, exist_ok=True); (d / fname).write_text(text, encoding="utf-8")

def table_results(tag="holdout"):
    cols = ["Accuracy", "Macro_Precision", "Macro_Recall", "Macro_F1", "Weighted_F1"]
    data = {n: load_metrics(tag, n) for _, n in available_rows(tag)}
    best = {c: max(m[c] for m in data.values()) for c in cols}
    L = [r"\begin{tabular}{llccccc}", r"\toprule", r"Type & Model & Acc. & Mac.\ P & Mac.\ R & Mac.\ F1 & Wtd.\ F1 \\", r"\midrule"]
    groups = [(g, [(l, n) for l, n in it if n in data]) for g, it in PAPER_LAYOUT]
    groups = [g for g in groups if g[1]]
    for gi, (g, items) in enumerate(groups):
        for k, (lab, n) in enumerate(items):
            cells = [g if k == 0 else "", lab] + [(r"\textbf{%s}" if abs(data[n][c] - best[c]) < 1e-12 else "%s") % pct(data[n][c]) for c in cols]
            L.append(" & ".join(cells) + r" \\")
        if gi < len(groups) - 1: L.append(r"\midrule")
    L += [r"\bottomrule", r"\end{tabular}"]
    write_tex(tag, "table_results.tex", "\n".join(L) + "\n")

def table_perclass(tag="holdout"):
    rows = available_rows(tag)
    L = [r"\begin{tabular}{lcccccc}", r"\toprule", "Model & " + " & ".join(CLASS_LABELS) + r" \\", r"\midrule"]
    for lab, n in rows:
        L.append(lab + " & " + " & ".join(f"{load_metrics(tag, n)['per_class'][c]['f1']:.2f}" for c in CLASSES) + r" \\")
    sup = load_metrics(tag, rows[0][1])["per_class"]
    L += [r"\midrule", "Test images & " + " & ".join(str(sup[c]["support"]) for c in CLASSES) + r" \\", r"\bottomrule", r"\end{tabular}"]
    write_tex(tag, "table_perclass.tex", "\n".join(L) + "\n")

def table_ablation(tag="holdout"):
    # (a) frozen head only (Phase 1) vs fine-tuned;  (b) ML classifiers on frozen vs fine-tuned features
    rows_a = []
    for m in CFG["MODELS"]:
        p, f = f"{m} (frozen, Phase 1 only)", m
        if all((OUT / tag / "metrics" / f"{slug(n)}.json").exists() for n in (p, f)):
            a, b = load_metrics(tag, p), load_metrics(tag, f)
            rows_a.append((m, a["Accuracy"], a["Macro_F1"], b["Accuracy"], b["Macro_F1"]))
    rows_b = []
    for bb in CFG["ML_BACKBONES"]:
        for clf in ("SVM", "KNN", "RF", "NB"):
            p, f = f"{bb} (frozen) + {clf}", f"{bb} (fine-tuned) + {clf}"
            if all((OUT / tag / "metrics" / f"{slug(n)}.json").exists() for n in (p, f)):
                a, b = load_metrics(tag, p), load_metrics(tag, f)
                rows_b.append((f"{bb} + {clf}", a["Accuracy"], a["Macro_F1"], b["Accuracy"], b["Macro_F1"]))
    pd.DataFrame(rows_a, columns=["Model", "Acc_frozen_only", "F1_frozen_only", "Acc_finetuned", "F1_finetuned"]).to_csv(OUT / tag / "metrics" / "ablation_finetuning.csv", index=False)
    pd.DataFrame(rows_b, columns=["Pipeline", "Acc_frozen_feat", "F1_frozen_feat", "Acc_finetuned_feat", "F1_finetuned_feat"]).to_csv(OUT / tag / "metrics" / "ablation_features.csv", index=False)
    L = [r"\begin{tabular}{lcccc}", r"\toprule", r" & \multicolumn{2}{c}{Frozen / frozen features} & \multicolumn{2}{c}{Fine-tuned / fine-tuned features} \\",
         r"Model & Acc. & Macro F1 & Acc. & Macro F1 \\", r"\midrule"]
    for r in rows_a: L.append(f"{r[0]} & {pct(r[1])} & {pct(r[2])} & {pct(r[3])} & {pct(r[4])} \\\\")
    if rows_a and rows_b: L.append(r"\midrule")
    for r in rows_b: L.append(f"{r[0]} & {pct(r[1])} & {pct(r[2])} & {pct(r[3])} & {pct(r[4])} \\\\")
    L += [r"\bottomrule", r"\end{tabular}"]
    write_tex(tag, "table_ablation.tex", "\n".join(L) + "\n")

def table_splits():
    L = [r"\begin{tabular}{lrrrr}", r"\toprule", r"Class & Fruit & Leaf & Total & Percentage \\", r"\midrule"]
    order = sorted(CLASSES, key=lambda c: -counts.loc[c, "Total"])
    for c in order:
        L.append(f"{c.replace('_', ' ')} & {int(counts.loc[c].get('Fruits', 0))} & {int(counts.loc[c].get('Leaves', 0))} & {int(counts.loc[c, 'Total'])} & {counts.loc[c, 'Percent']:.1f}\\% \\\\")
    L += [r"\midrule", f"Total & {int(counts.get('Fruits', pd.Series(0)).sum())} & {int(counts.get('Leaves', pd.Series(0)).sum())} & {int(counts['Total'].sum())} & 100\\% \\\\", r"\bottomrule", r"\end{tabular}"]
    write_tex("holdout", "table_dataset.tex", "\n".join(L) + "\n")
    L = [r"\begin{tabular}{lrrr}", r"\toprule", r"Class & Train & Validation & Test \\", r"\midrule"]
    for c in CLASSES:
        L.append(f"{c.replace('_', ' ')} & {split_counts.loc[c, 'train']} & {split_counts.loc[c, 'val']} & {split_counts.loc[c, 'test']} \\\\")
    L += [r"\midrule", f"Total & {split_counts.loc['Total', 'train']} & {split_counts.loc['Total', 'val']} & {split_counts.loc['Total', 'test']} \\\\", r"\bottomrule", r"\end{tabular}"]
    write_tex("holdout", "table_split.tex", "\n".join(L) + "\n")

In [ ]:
def make_paper_assets(tag="holdout"):
    summary = collect_summary(tag)
    fig_class_distribution(); fig_training_curves(tag)
    cm_grid(PAPER_ROWS[:4], 2, "fig_cm_dl", tag)                       # four fine-tuned CNNs
    cm_grid(PAPER_ROWS[4:8], 2, "fig_cm_ml", tag)                      # classical classifiers on frozen ResNet50 features
    cm_grid([("DenseNet121 + SVM", "DenseNet121 (frozen) + SVM"), ("DenseNet121 + RF", "DenseNet121 (frozen) + RF"),
             ("Soft-voting ensemble", "ResNet50 + DenseNet121 (soft voting)")], 3, "fig_cm_hybrid", tag, width=7.6, aspect=1.12)
    fig_comparison(tag); fig_perclass_f1(tag)
    table_results(tag); table_perclass(tag); table_ablation(tag); table_splits()
    return summary

summary = make_paper_assets("holdout")
display(summary[["name", "Accuracy", "Macro_Precision", "Macro_Recall", "Macro_F1", "Weighted_F1"]].round(4))

## 10. Organ-wise accuracy (Is the model recognising the disease or the organ?)
Scab occurs only in fruit images and Melanose only in leaf images, so the label is confounded with the organ for these classes. Accuracy per organ is saved to `holdout/metrics/organ_accuracy.csv`.

In [ ]:
def organ_analysis(tag="holdout"):
    rows = []
    for lab, n in available_rows(tag):
        _, _, d = load_preds(tag, n)
        r = {"Model": lab}
        for dom in ("Fruits", "Leaves"):
            s = d[d["domain"] == dom]
            r[f"Acc_{dom}"] = float((s["y_true"] == s["y_pred"]).mean()) if len(s) else np.nan; r[f"n_{dom}"] = int(len(s))
        rows.append(r)
    out = pd.DataFrame(rows); out.to_csv(OUT / tag / "metrics" / "organ_accuracy.csv", index=False)
    fig, ax = plt.subplots(figsize=(7.0, 3.0)); x = np.arange(len(out)); w = 0.38
    ax.bar(x - w / 2, 100 * out["Acc_Fruits"], w, label=f"Fruit images (n={out['n_Fruits'].iloc[0]})", color=OKABE[1], edgecolor="black", linewidth=0.3)
    ax.bar(x + w / 2, 100 * out["Acc_Leaves"], w, label=f"Leaf images (n={out['n_Leaves'].iloc[0]})", color=OKABE[2], edgecolor="black", linewidth=0.3)
    ax.set_xticks(x); ax.set_xticklabels(out["Model"], rotation=30, ha="right"); ax.set_ylabel("Test accuracy (%)"); ax.set_ylim(0, 105)
    ax.legend(frameon=False, ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.0)); ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)
    fig.tight_layout(); save_fig(fig, "fig_organ_accuracy", tag)
    return out

display(organ_analysis("holdout").round(3))

## 11. Grad-CAM for the final ResNet50 (what does the network look at?)
Rows: a correctly classified test image per class, its Grad-CAM, and (if one exists) a misclassified test image of that class. The heat map explains the *predicted* class. This is a qualitative aid only.

In [ ]:
def load_img_array(path, img_size):
    # same loading and resizing (nearest) as the Keras generators, so the prediction is identical to the stored one
    return np.asarray(keras.utils.load_img(path, target_size=img_size, interpolation="nearest"), dtype=np.float32)

def gradcam(model, x_pre, class_idx=None):
    base = [l for l in model.layers if isinstance(l, keras.Model)][0]
    dense = model.get_layer("pred")
    x = tf.convert_to_tensor(x_pre[None], dtype=tf.float32)
    conv = base(x, training=False)
    with tf.GradientTape() as tape:
        tape.watch(conv)
        pooled = tf.reduce_mean(conv, axis=(1, 2))
        logits = tf.matmul(pooled, dense.kernel) + dense.bias          # pre-softmax score
        if class_idx is None:
            class_idx = int(tf.argmax(logits[0]))
        score = logits[:, class_idx]
    grads = tape.gradient(score, conv)
    w = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(w * conv, axis=-1))[0].numpy()
    return cam / (cam.max() + 1e-8), class_idx

def overlay(img_uint8, cam, alpha=0.40):
    cam_r = tf.image.resize(cam[..., None], img_uint8.shape[:2], method="bilinear").numpy()[..., 0]
    heat = matplotlib.colormaps["jet"](cam_r)[..., :3]
    return np.clip((1 - alpha) * img_uint8 / 255.0 + alpha * heat, 0, 1)

def fig_gradcam(tag="holdout", name=None):
    name = name or CFG["GRADCAM_MODEL"]; mpath = OUT / tag / "models" / f"{name}_final.keras"
    if not mpath.exists():
        print("Grad-CAM skipped: model file not found:", mpath); return
    model = keras.models.load_model(mpath); img = CFG["IMG"][name]; _, pre = APPS[name]
    _, _, d = load_preds(tag, name)
    n_check = [0, 0]                                                    # images checked, prediction mismatches
    fig, axes = plt.subplots(3, len(CLASSES), figsize=(7.2, 3.6)); [ax.axis("off") for ax in axes.ravel()]
    for j, c in enumerate(CLASSES):
        axes[0, j].set_title(CLASS_LABELS[j], fontsize=8)
        ok = d[(d["y_true"] == j) & (d["y_pred"] == j)]; bad = d[(d["y_true"] == j) & (d["y_pred"] != j)]
        for row, sample in ((0, ok), (2, bad)):
            if len(sample) == 0: continue
            rec = sample.iloc[0]; arr = load_img_array(rec["filepath"], img)
            k = int(rec["y_pred"])                                        # explain the stored prediction
            cam, k_model = gradcam(model, pre(arr.copy()), class_idx=k)
            n_check[0] += 1; n_check[1] += int(int(np.argmax(model.predict(pre(arr.copy())[None], verbose=0))) != k)
            if row == 0:
                axes[0, j].imshow(arr.astype(np.uint8))
                axes[1, j].imshow(overlay(arr, cam))
            else:
                axes[2, j].imshow(overlay(arr, cam)); axes[2, j].set_title(f"pred: {CLASS_LABELS[k]}", fontsize=6.5, color="firebrick")
    for r, t in enumerate(("Image (correct)", "Grad-CAM (correct)", "Grad-CAM (misclassified)")):
        axes[r, 0].text(-0.08, 0.5, t, transform=axes[r, 0].transAxes, rotation=90, va="center", ha="right", fontsize=7)
    print(f"Grad-CAM sanity check: {n_check[1]} of {n_check[0]} re-computed predictions differ from the stored predictions (should be 0).")
    fig.tight_layout(); save_fig(fig, f"fig_gradcam_{name.lower()}", tag)
    del model; keras.backend.clear_session()

if RUN_GRADCAM:
    fig_gradcam("holdout")

## 12. Optional: stratified k-fold cross-validation (recommended for the final paper)
With only 114 test images (2 each for Melanose and Scab) a single split is fragile. Set `RUN_CV = True` in the configuration cell to repeat the whole experiment on `CV_FOLDS` stratified folds. Outer test folds are disjoint, so the pooled predictions cover **all 759 images** (13 Melanose, 15 Scab). Expect roughly `CV_FOLDS` times the training time of the hold-out run; each fold is cached, so an interrupted run resumes. Fold models are deleted after each fold to save disk space.

In [ ]:
KEEP_CV_MODELS = False
CV_MODELS = CFG["MODELS"] if not SMOKE_TEST else ["ResNet50", "DenseNet121"]
CV_NAMES = [n for _, n in PAPER_ROWS]

def fold_complete(k):
    # a fold is complete when its last stage (the ensemble) has been saved
    last = "ResNet50 + DenseNet121 (soft voting)" if {"ResNet50", "DenseNet121"} <= set(CV_MODELS) else None
    p = OUT / f"cv_fold{k}" / "metrics"
    return (p / f"{slug(last)}.json").exists() if last else (p.exists() and any(p.glob("*.json")))

def run_cv():
    manifest = []
    for fold, tr, va, te in cv_splits(df, CV_FOLDS):
        k = fold + 1; tag = f"cv_fold{k}"
        manifest.append(pd.concat([tr.assign(split="train"), va.assign(split="val"), te.assign(split="test")]).assign(fold=k))
        if CV_RUN_FOLDS is not None and k not in CV_RUN_FOLDS:
            continue
        print(f"\n########## {tag} ##########")
        run_experiment(tag, tr, va, te, models=CV_MODELS)
        if not KEEP_CV_MODELS:
            for f in (OUT / tag / "models").glob("*"): f.unlink()
        log_mem(tag)
    done_folds = [k for k in range(1, CV_FOLDS + 1) if fold_complete(k)]
    print("Complete folds:", done_folds, "of", CV_FOLDS)
    if len(done_folds) < CV_FOLDS:
        print("WARNING: not all folds are complete; the cross-validation summary below covers only the complete folds.")
    per_fold = []
    for k in done_folds:
        s = collect_summary(f"cv_fold{k}"); s["fold"] = k; per_fold.append(s)
    if not per_fold:
        print("No complete fold yet."); return None, None
    D = OUT / "cv_summary"; D.mkdir(exist_ok=True); (D / "figures").mkdir(exist_ok=True); (D / "tables").mkdir(exist_ok=True)
    pd.concat(manifest).to_csv(OUT / "data" / "cv_split_manifest.csv", index=False)
    allf = pd.concat(per_fold, ignore_index=True); allf.to_csv(D / "cv_per_fold_metrics.csv", index=False)
    cols = ["Accuracy", "Macro_Precision", "Macro_Recall", "Macro_F1", "Weighted_F1"]
    stat = allf.groupby("name")[cols].agg(["mean", "std"]); stat.columns = [f"{a}_{b}" for a, b in stat.columns]
    stat.to_csv(D / "cv_mean_sd.csv")
    pooled_rows, pooled_cm = [], {}
    for name in sorted(allf["name"].unique()):
        yt, yp = [], []
        for k in done_folds:
            p = OUT / f"cv_fold{k}" / "preds" / f"{slug(name)}.csv"
            if p.exists(): d = pd.read_csv(p); yt.append(d["y_true"].values); yp.append(d["y_pred"].values)
        if yt:
            m = compute_all_metrics(np.concatenate(yt), np.concatenate(yp))
            pooled_cm[name] = m["confusion_matrix"]
            pooled_rows.append({"name": name, **{k: m[k] for k in cols}, "n": m["n_test"], **{f"F1_{c}": m["per_class"][c]["f1"] for c in CLASSES},
                                **{f"support_{c}": m["per_class"][c]["support"] for c in CLASSES}})
    pd.DataFrame(pooled_rows).to_csv(D / "cv_pooled_metrics.csv", index=False)
    json.dump(pooled_cm, open(D / "cv_pooled_confusion.json", "w"))
    cv_table(stat); cv_figures(allf, pooled_cm)
    return allf, stat

def cv_table(stat):
    L = [r"\begin{tabular}{lcccc}", r"\toprule", r"Model & Accuracy & Macro F1 & Weighted F1 & Macro recall \\", r"\midrule"]
    for lab, n in PAPER_ROWS:
        if n in stat.index:
            r = stat.loc[n]; f = lambda k: f"{100 * r[k + '_mean']:.1f} $\\pm$ {100 * r[k + '_std']:.1f}"
            L.append(f"{lab} & {f('Accuracy')} & {f('Macro_F1')} & {f('Weighted_F1')} & {f('Macro_Recall')} \\\\")
    L += [r"\bottomrule", r"\end{tabular}"]
    write_tex("cv_summary", "cv_table.tex", "\n".join(L) + "\n")

def cv_figures(allf, pooled_cm):
    rows = [(l, n) for l, n in PAPER_ROWS if n in set(allf["name"])]
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.3), sharey=False)
    for ax, (key, title) in zip(axes, (("Accuracy", "Accuracy"), ("Macro_F1", "Macro F1-score"))):
        data = [100 * allf[allf["name"] == n][key].values for _, n in rows]
        try:
            ax.boxplot(data, orientation="horizontal", widths=0.55, showfliers=False, medianprops=dict(color="black"))   # Matplotlib >= 3.10
        except TypeError:
            ax.boxplot(data, vert=False, widths=0.55, showfliers=False, medianprops=dict(color="black"))               # older Matplotlib
        for i, v in enumerate(data):
            ax.scatter(v, np.full(len(v), i + 1) + np.random.default_rng(0).uniform(-0.12, 0.12, len(v)), s=10, color=OKABE[0], zorder=3)
        ax.set_yticks(range(1, len(rows) + 1)); ax.set_yticklabels([l for l, _ in rows] if ax is axes[0] else [], fontsize=8)
        ax.set_xlabel(f"{title} (%)"); ax.invert_yaxis(); ax.xaxis.grid(True, alpha=0.3)
    fig.tight_layout(); save_fig(fig, "fig_cv_boxplot", "cv_summary")
    items = [(l, n) for l, n in rows if n in pooled_cm and l in ("EfficientNetB0", "ResNet50", "DenseNet121", "InceptionV3", "ResNet50 + DenseNet121")]
    if items:
        ncols = 3; nrows = math.ceil(len(items) / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(7.2, 2.5 * nrows), squeeze=False); [a.axis("off") for a in axes.ravel()]
        for ax, (l, n) in zip(axes.ravel(), items):
            ax.axis("on"); im = draw_cm(ax, pooled_cm[n], l)
        fig.tight_layout(); save_fig(fig, "fig_cv_cm_pooled", "cv_summary")

if RUN_CV:
    cv_all, cv_stat = run_cv()
    if cv_stat is not None:
        display(cv_stat.round(4))
else:
    print("RUN_CV = False: cross-validation skipped.")

## 13. Run information, README and download package

In [ ]:
import sklearn
run_info = dict(created=datetime.datetime.now().isoformat(timespec="seconds"), python=platform.python_version(),
                tensorflow=tf.__version__, keras=keras.__version__, numpy=np.__version__, pandas=pd.__version__,
                scikit_learn=sklearn.__version__, matplotlib=matplotlib.__version__,
                gpus=[g.name for g in tf.config.list_physical_devices("GPU")], seed=SEED, smoke_test=SMOKE_TEST,
                n_images=int(len(df)), split_sizes=dict(train=len(train_df), val=len(val_df), test=len(test_df)),
                run_cv=RUN_CV, cv_folds=CV_FOLDS if RUN_CV else None, config=cfg_now)
json.dump(run_info, open(OUT / "run_info.json", "w"), indent=2)

README = '''OUTPUT GUIDE
============
paper_assets/figures   600-dpi PNG + vector PDF, same file names as the paper
    fig_class_dist, fig_curves, fig_cm_dl, fig_cm_ml, fig_cm_hybrid, fig_comparison      (paper figures)
    fig_perclass_f1, fig_organ_accuracy, fig_gradcam_resnet50                            (new, optional)
    fig_cv_boxplot, fig_cv_cm_pooled                                                    (only if RUN_CV = True)
paper_assets/tables    LaTeX tabular files, paste with \\input{tables/<file>.tex}
    table_dataset, table_split, table_results, table_perclass, table_ablation, cv_table
holdout/metrics        summary.csv (all models), one JSON per model (metrics, per-class results, confusion matrix),
                       ablation_finetuning.csv, ablation_features.csv, organ_accuracy.csv
holdout/preds          test-set predictions and probabilities for every model (CSV)
holdout/history        training histories (CSV) and training info (epochs, best validation loss, time)
holdout/features       extracted deep features (NPZ) for the ML classifiers
holdout/models         fine-tuned models (*.keras)
cv_fold*/              the same structure for every cross-validation fold
cv_summary             cv_per_fold_metrics.csv, cv_mean_sd.csv, cv_pooled_metrics.csv, cv_pooled_confusion.json
data                   dataset manifest, class x organ counts, split manifests, class weights
run_info.json, config.json   software versions, GPU, seed and the full configuration of this run
'''
(OUT / "README_outputs.txt").write_text(README, encoding="utf-8")

def make_zip(zip_path=None):
    zip_path = Path(zip_path or OUT.parent / "results_package.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in OUT.rglob("*"):
            if p.is_file() and p.suffix != ".keras" and not p.name.endswith(".weights.h5"):
                z.write(p, p.relative_to(OUT.parent))
    return zip_path

zp = make_zip()
print("Package (no model files):", zp, f"{zp.stat().st_size / 1e6:.1f} MB")
print("Key files:")
for p in sorted((OUT / "paper_assets").rglob("*")):
    if p.is_file() and p.suffix in (".png", ".tex"): print("  ", p.relative_to(OUT))

## 12b. Cross-validation analyses from the saved results (no training)
Pooled per-class results over all test folds (every image is a test image exactly once), accuracy by organ, the effect of fine-tuning and of the feature source (mean +/- SD over folds), 95 % Wilson intervals for the pooled accuracy and exact McNemar tests on the pooled paired predictions. Pooled tests treat the images as independent although the training sets of the folds overlap, so the p-values are descriptive only. Outputs: `cv_summary/metrics/*.csv`, `cv_summary/tables/*.tex`, `cv_summary/figures/*`.

In [ ]:
from scipy.stats import binomtest

def wilson_ci(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); m = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - m) / d, (c + m) / d

CV_PAIRS = [("ResNet50 (frozen) + SVM", "ResNet50"), ("ResNet50 (frozen) + SVM", "DenseNet121 (frozen) + SVM"),
            ("ResNet50 (frozen) + SVM", "ResNet50 + DenseNet121 (soft voting)"), ("ResNet50", "ResNet50 + DenseNet121 (soft voting)"),
            ("DenseNet121 (frozen) + SVM", "ResNet50"), ("ResNet50", "EfficientNetB0"), ("ResNet50", "DenseNet121"), ("ResNet50", "InceptionV3")]

def pm(series_mean, series_sd):
    return f"{100 * series_mean:.1f} $\\pm$ {100 * series_sd:.1f}"

def cv_extra_analyses():
    tags = [f"cv_fold{k}" for k in range(1, CV_FOLDS + 1) if fold_complete(k)]
    assert tags, "no complete fold"
    D = OUT / "cv_summary"; [(D / s).mkdir(parents=True, exist_ok=True) for s in ("metrics", "tables", "figures")]
    rows = [(l, n) for l, n in PAPER_ROWS if all((OUT / t / "preds" / f"{slug(n)}.csv").exists() for t in tags)]
    def pooled(name):
        ds = []
        for t in tags:
            d = pd.read_csv(OUT / t / "preds" / f"{slug(name)}.csv"); d["fold"] = int(t.replace("cv_fold", "")); ds.append(d)
        return pd.concat(ds, ignore_index=True)
    P = {n: pooled(n) for _, n in rows}
    per_fold = pd.concat([collect_summary(t).assign(fold=int(t.replace("cv_fold", ""))) for t in tags]); per_fold = per_fold.set_index(["name", "fold"])

    # 1. pooled per-class results
    pc = []
    for lab, n in rows:
        m = compute_all_metrics(P[n]["y_true"].values, P[n]["y_pred"].values)
        pc.append({"Model": lab, "pooled_accuracy": m["Accuracy"], "pooled_macro_F1": m["Macro_F1"],
                   **{f"F1_{c}": m["per_class"][c]["f1"] for c in CLASSES}, **{f"recall_{c}": m["per_class"][c]["recall"] for c in CLASSES},
                   **{f"precision_{c}": m["per_class"][c]["precision"] for c in CLASSES}, **{f"support_{c}": m["per_class"][c]["support"] for c in CLASSES}})
    pc = pd.DataFrame(pc); pc.to_csv(D / "metrics" / "cv_perclass_pooled.csv", index=False)
    L = [r"\begin{tabular}{lcccccc}", r"\toprule", "Model & " + " & ".join(CLASS_LABELS) + r" \\", r"\midrule"]
    for _, r in pc.iterrows(): L.append(r["Model"] + " & " + " & ".join(f"{r['F1_' + c]:.2f}" for c in CLASSES) + r" \\")
    L += [r"\midrule", "Images & " + " & ".join(str(int(pc.iloc[0]["support_" + c])) for c in CLASSES) + r" \\", r"\bottomrule", r"\end{tabular}"]
    write_tex("cv_summary", "cv_perclass.tex", "\n".join(L) + "\n")

    # 2. accuracy by organ (pooled, Wilson intervals)
    og = []
    for lab, n in rows:
        r = {"Model": lab}
        for dom in ("Fruits", "Leaves"):
            s = P[n][P[n]["domain"] == dom]; k = int((s["y_true"] == s["y_pred"]).sum()); lo, hi = wilson_ci(k, len(s))
            r.update({f"acc_{dom}": k / len(s), f"lo_{dom}": lo, f"hi_{dom}": hi, f"n_{dom}": len(s)})
        og.append(r)
    og = pd.DataFrame(og); og.to_csv(D / "metrics" / "cv_organ_accuracy.csv", index=False)
    fig, ax = plt.subplots(figsize=(7.2, 3.2)); x = np.arange(len(og)); w = 0.38
    for off, dom, col, nm in ((-w / 2, "Fruits", OKABE[1], "Fruit"), (w / 2, "Leaves", OKABE[2], "Leaf")):
        a = 100 * og[f"acc_{dom}"].values
        err = np.clip(np.vstack([a - 100 * og[f"lo_{dom}"].values, 100 * og[f"hi_{dom}"].values - a]), 0, None)   # rounding can give -1e-15 at 0 % or 100 %
        ax.bar(x + off, a, w, yerr=err, capsize=1.5, color=col, edgecolor="black", linewidth=0.3, error_kw=dict(lw=0.6), label=f"{nm} images (n={int(og['n_' + dom].iloc[0])})")
    ax.set_xticks(x); ax.set_xticklabels(og["Model"], rotation=35, ha="right"); ax.set_ylabel("Pooled accuracy (%)"); ax.set_ylim(0, 105)
    ax.legend(frameon=False, ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.0)); ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)
    fig.tight_layout(); save_fig(fig, "fig_cv_organ_accuracy", "cv_summary")

    # 3. effect of fine-tuning and of the feature source (mean +/- SD over folds)
    def ms(name, col):
        v = per_fold.loc[name, col].values; return v.mean(), v.std(ddof=1)
    A = []; Lb = [r"\begin{tabular}{lccccc}", r"\toprule", r" & \multicolumn{2}{c}{Frozen / frozen features} & \multicolumn{2}{c}{Fine-tuned / fine-tuned features} & Phase 2 \\",
                  r"Model & Acc. & Macro F1 & Acc. & Macro F1 & selected \\", r"\midrule"]
    for m in CFG["MODELS"]:
        a, b = f"{m} (frozen, Phase 1 only)", m
        if a in per_fold.index.get_level_values(0) and b in per_fold.index.get_level_values(0):
            ph2 = sum(json.load(open(OUT / t / "history" / f"{m}_train_info.json"))["final_model_from"] == "phase2" for t in tags)
            r = [m, *ms(a, "Accuracy"), *ms(a, "Macro_F1"), *ms(b, "Accuracy"), *ms(b, "Macro_F1"), ph2]
            A.append(r); Lb.append(f"{m} & {pm(*ms(a, 'Accuracy'))} & {pm(*ms(a, 'Macro_F1'))} & {pm(*ms(b, 'Accuracy'))} & {pm(*ms(b, 'Macro_F1'))} & {ph2} of {len(tags)} \\\\")
    Lb.append(r"\midrule")
    for bb in CFG["ML_BACKBONES"]:
        for clf in ("SVM", "KNN", "RF", "NB"):
            a, b = f"{bb} (frozen) + {clf}", f"{bb} (fine-tuned) + {clf}"
            if a in per_fold.index.get_level_values(0) and b in per_fold.index.get_level_values(0):
                A.append([f"{bb} + {clf}", *ms(a, "Accuracy"), *ms(a, "Macro_F1"), *ms(b, "Accuracy"), *ms(b, "Macro_F1"), np.nan])
                Lb.append(f"{bb} + {clf} & {pm(*ms(a, 'Accuracy'))} & {pm(*ms(a, 'Macro_F1'))} & {pm(*ms(b, 'Accuracy'))} & {pm(*ms(b, 'Macro_F1'))} & -- \\\\")
    Lb += [r"\bottomrule", r"\end{tabular}"]
    pd.DataFrame(A, columns=["Model", "Acc_frozen_mean", "Acc_frozen_sd", "F1_frozen_mean", "F1_frozen_sd", "Acc_final_mean", "Acc_final_sd", "F1_final_mean", "F1_final_sd", "phase2_selected"]).to_csv(D / "metrics" / "cv_ablation.csv", index=False)
    write_tex("cv_summary", "cv_ablation.tex", "\n".join(Lb) + "\n")

    # 4. pooled accuracy with Wilson interval, exact McNemar tests on pooled paired predictions
    ci = []
    for lab, n in rows:
        d = P[n]; k = int((d["y_true"] == d["y_pred"]).sum()); lo, hi = wilson_ci(k, len(d)); ci.append({"Model": lab, "correct": k, "n": len(d), "pooled_accuracy": k / len(d), "ci_low": lo, "ci_high": hi})
    pd.DataFrame(ci).to_csv(D / "metrics" / "cv_pooled_accuracy_ci.csv", index=False)
    tests = []
    for a, b in CV_PAIRS:
        if a in per_fold.index.get_level_values(0) and b in per_fold.index.get_level_values(0):
            A_, B_ = pooled(a), pooled(b); assert (A_["filepath"].values == B_["filepath"].values).all()
            ca, cb = (A_["y_true"] == A_["y_pred"]).values, (B_["y_true"] == B_["y_pred"]).values
            n10, n01 = int((ca & ~cb).sum()), int((~ca & cb).sum())
            diffs = per_fold.loc[a, "Accuracy"].values - per_fold.loc[b, "Accuracy"].values
            tests.append({"A": a, "B": b, "only_A_correct": n10, "only_B_correct": n01, "p_exact_mcnemar": binomtest(min(n10, n01), n10 + n01, 0.5).pvalue if n10 + n01 else 1.0,
                          "acc_diff_pooled": float(ca.mean() - cb.mean()), "acc_diff_fold_mean": float(diffs.mean()), "acc_diff_fold_sd": float(diffs.std(ddof=1))})
    pd.DataFrame(tests).to_csv(D / "metrics" / "cv_mcnemar_pooled.csv", index=False)

    # 5. main table for the paper: mean +/- SD over folds, pooled accuracy [95 % CI], hold-out accuracy
    S = per_fold[["Accuracy", "Macro_F1", "Weighted_F1"]].groupby(level=0).agg(["mean", "std"]); best = {c: max(S.loc[n, (c, "mean")] for _, n in rows) for c in ("Accuracy", "Macro_F1", "Weighted_F1")}
    Lm = [r"\begin{tabular}{llccccc}", r"\toprule", r"Type & Model & Accuracy & Macro F1 & Weighted F1 & Pooled acc.\ [95\% CI] & Hold-out acc. \\", r"\midrule"]
    groups = [(g, [(l, n) for l, n in it if n in dict(rows).values()]) for g, it in PAPER_LAYOUT]; groups = [g for g in groups if g[1]]
    cidx = {r["Model"]: r for r in ci}
    for gi, (g, items) in enumerate(groups):
        for k, (lab, n) in enumerate(items):
            cells = [g if k == 0 else "", lab]
            for c in ("Accuracy", "Macro_F1", "Weighted_F1"):
                t = pm(S.loc[n, (c, "mean")], S.loc[n, (c, "std")]); cells.append(r"\textbf{" + t + "}" if abs(S.loc[n, (c, "mean")] - best[c]) < 1e-12 else t)
            r_ = cidx[lab]; cells.append(f"{100 * r_['pooled_accuracy']:.1f} [{100 * r_['ci_low']:.1f}, {100 * r_['ci_high']:.1f}]")
            hp = OUT / "holdout" / "metrics" / f"{slug(n)}.json"; cells.append(f"{100 * json.load(open(hp))['Accuracy']:.1f}" if hp.exists() else "--")
            Lm.append(" & ".join(cells) + r" \\")
        if gi < len(groups) - 1: Lm.append(r"\midrule")
    Lm += [r"\bottomrule", r"\end{tabular}"]
    write_tex("cv_summary", "cv_table_paper.tex", "\n".join(Lm) + "\n")

    # 6. pooled confusion matrices (all test images of all folds)
    sel = [(l, n) for l, n in rows if l in ("EfficientNetB0", "ResNet50", "DenseNet121", "InceptionV3", "ResNet50 + SVM", "DenseNet121 + SVM", "ResNet50 + DenseNet121")]
    ncols = 3; nrows = math.ceil(len(sel) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(7.2, 2.5 * nrows), squeeze=False); [a.axis("off") for a in axes.ravel()]
    for ax, (l, n) in zip(axes.ravel(), sel):
        ax.axis("on"); cm = confusion_matrix(P[n]["y_true"], P[n]["y_pred"], labels=list(range(len(CLASSES)))); draw_cm(ax, cm, l if l != "ResNet50 + DenseNet121" else "Soft-voting ensemble")
    fig.tight_layout(); save_fig(fig, "fig_cv_cm_pooled", "cv_summary")
    print("cross-validation analyses written for folds:", [t.replace("cv_fold", "") for t in tags])

if any(fold_complete(k) for k in range(1, CV_FOLDS + 1)):
    cv_extra_analyses()